# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

### Checkpoint 1 — Môi trường, dữ liệu và nhãn
Chọn **T4 GPU** trong Runtime → Change runtime type. Chạy lần lượt các cell đến mốc dừng Checkpoint 1.
Mã nguồn được clone vào `/content`; dữ liệu và trọng số không đưa vào Git. Drive giữ bản dự phòng khi Colab ngắt.


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/Nituv05/K4-Track-4-Day10-VuThuongTin-2A202602955-VisionLanguageModel.git"
REPO_DIR = Path("/content/road-damage-lab")
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
# Nếu notebook/mã Checkpoint 1 chưa có trong clone: push từ máy trước,
# rồi chạy `git pull --ff-only` ở Colab và chạy lại cell này.
if not (REPO_DIR / "student/data.py").exists():
    raise RuntimeError("Push mã Checkpoint 1 lên GitHub rồi cập nhật clone Colab trước khi chạy.")
subprocess.run(["nvidia-smi", "-L"], check=True)


In [ ]:
%pip -q install -r requirements-colab.txt pytest
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Chưa có GPU. Hãy chọn runtime T4 và chạy lại.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
subprocess.run(["python", "-m", "pytest", "-q", "tests", "student/tests"], check=True)


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/LabDay10_VisionLanguageModel")
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Nơi lưu checkpoint:", DRIVE_RUN_DIR)


In [ ]:
# Chọn đúng một nguồn. ZIP/SOURCE là đường dẫn trên Colab hoặc Google Drive.
DATA_MODE = "kaggle"  # "kaggle", "zip", hoặc "source"
DATA_ZIP = ""         # Ví dụ: /content/drive/MyDrive/road-damage.zip
DATA_SOURCE = ""      # Thư mục có images/ và labels-YOLO/

if DATA_MODE == "kaggle":
    from google.colab import userdata
    try:
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        raise RuntimeError("Thêm KAGGLE_USERNAME và KAGGLE_KEY vào Colab Secrets, bật quyền notebook; hoặc chọn zip/source.") from None
    if not os.environ.get("KAGGLE_USERNAME") or not os.environ.get("KAGGLE_KEY"):
        raise RuntimeError("Colab Secrets Kaggle đang trống.")
# Không in giá trị Secrets và không ghi chúng vào notebook.


## 1. Dữ liệu: tải và chia split

In [ ]:
if DATA_MODE == "kaggle":
    data_args = ["--kaggle"]
elif DATA_MODE == "zip" and Path(DATA_ZIP).is_file():
    data_args = ["--zip", DATA_ZIP]
elif DATA_MODE == "source" and Path(DATA_SOURCE).is_dir():
    data_args = ["--source", DATA_SOURCE]
else:
    raise ValueError("Kiểm tra DATA_MODE và đường dẫn nguồn dữ liệu.")
subprocess.run(["python", "scripts/prepare_data.py", *data_args], check=True)


In [ ]:
from student.data import write_checkpoint1
import json

cp1_summary = write_checkpoint1()
print(json.dumps(cp1_summary, ensure_ascii=False, indent=2))


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from student.data import PROMPT, objects_to_answer, inspect_splits
from IPython.display import display
from PIL import Image

records, _ = inspect_splits()
example = records["train"][0]
print("Prompt:", PROMPT)
print("Image:", example["image_id"])
print("Answer:", objects_to_answer(example["objects"]))
# Trái: box YOLO gốc; phải: JSON 0–1000 quy ngược về pixel.
for image_path in sorted(Path("work/checkpoint1").glob("labels_*.png")):
    with Image.open(image_path) as image:
        display(image.copy())


In [ ]:
import shutil
from google.colab import files

backup_dir = DRIVE_RUN_DIR / "checkpoint1"
shutil.copytree("work/checkpoint1", backup_dir, dirs_exist_ok=True)
shutil.copy2("data/splits/meta.json", backup_dir / "split_meta.json")
archive = shutil.make_archive("/content/checkpoint1", "zip", backup_dir)
print("Đã sao lưu:", backup_dir)
files.download(archive)


### DỪNG — Checkpoint 1
Kiểm tra sáu ảnh: box ở hai phía phải khớp (sai số làm tròn tối đa 0,32 pixel theo chiều rộng).
Gửi `checkpoint1.zip` và log pytest để kiểm chứng dữ liệu thật. Đây là ảnh **nhãn**, chưa phải sáu ảnh dự đoán đúng/sai cần cho báo cáo cuối.
Nếu đạt, tự commit/push Checkpoint 1. Chỉ tiếp tục E0 sau khi đã xác nhận mốc này.
Các phần phía dưới sẽ được hoàn thiện ở Checkpoint tiếp theo.


## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9